# Extracción de Datos
Tenemos diferentes conjuntos de datos, para los datos satélitales tenemos 2 archivos csv, para la temperatura, precipitación, elevación y pendiente tenemos diferentes tipos de rasters, y tenemos shape files que nos dan la geometría de la parcela. Para la temperatura y elevación tenemos rasters para cada mes desde del 2022 a 2025. En caso de la elevación y pendiente es solo un raster.

El objetivo es de extraer los datos para crear un solo archivo csv.

In [2]:
import pandas as pd 
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from rasterio.plot import plotting_extent
from rasterio.windows import from_bounds
import geopandas as gpd
import matplotlib.pyplot as plt
from pyproj import Transformer
import shapely
from rasterio.mask import mask
from shapely.geometry import mapping
from shapely.geometry import box
from rasterio.transform import xy
from collections import Counter
from pathlib import Path

In [3]:
parcelas_gdf = gpd.read_file('../../DataSet/DATASET_RETO_AGRO_2026/Parcelas_Reto_AGC_CONJUNTO_70_30/Parcelas_Reto_AGC_CONJUNTO.shp')

parcelas_gdf.head()

,Cultivo,ID_POLIGON,área_ha,Municipio,Estado,CONJUNTO,geometry
0,Cebada,AGC_196,8.57,Emiliano Zapata,Hidalgo,ENTRENAMIENTO,"POLYGON Z ((-98.57739 19.65802 0, -98.57751 19..."
1,Cebada,AGC_044,0.68,Emiliano Zapata,Hidalgo,ENTRENAMIENTO,"POLYGON Z ((-98.58655 19.62943 0, -98.5862 19...."
2,Cebada,AGC_020,8.34,Emiliano Zapata,Hidalgo,ENTRENAMIENTO,"POLYGON Z ((-98.58494 19.6278 0, -98.58656 19...."
3,CEBADA,AGC_174,6.89,Cuautepec de Hinojosa,Hidalgo,PREDICCION,"POLYGON Z ((-98.30604 19.9 0, -98.30486 19.899..."
4,CEBADA,AGC_083,2.87,Cuautepec de Hinojosa,Hidalgo,ENTRENAMIENTO,"POLYGON Z ((-98.29383 19.8882 0, -98.29378 19...."


In [12]:
print(f'Shape: {parcelas_gdf.shape}')
print(f'CRS: {parcelas_gdf.crs}')
print(f'Columns: {parcelas_gdf.columns}')

Shape: (197, 7)
CRS: EPSG:4326
Columns: Index(['Cultivo', 'ID_POLIGON', 'área_ha', 'Municipio', 'Estado', 'CONJUNTO',
       'geometry'],
      dtype='str')


## Datos Topográficos.

### Raster de Elevación.

In [13]:
elev_path = '../../DataSet/DATASET_RETO_AGRO_2026/Reto_AgroCebada_Topografia_INEGI_CEM4/Reto_AgroCebada_Topografia_INEGI_CEM4/Reto_AgroCebada_Topografia_INEGI_CEM4/Elevacion_INEGI_CEM4_120m.tif'

print('Elevación:')

with rasterio.open(elev_path) as src_elev:
    print(f'CSR: {src_elev.crs}')
    print(f'Dimensiones: {src_elev.width} x {src_elev.height}')
    print(f'Bandas: {src_elev.count}')
    print(f'Resolución: {src_elev.res}')
    print(f'Bounds: {src_elev.bounds}')
    print(f'No Data: {src_elev.nodata}')
    print(f'Data type: {src_elev.dtypes[0]}')

Elevación:
CSR: EPSG:6372
Dimensiones: 2900 x 3362
Bandas: 1
Resolución: (120.0, 120.0)
Bounds: BoundingBox(left=2715840.0, bottom=650400.0, right=3063840.0, top=1053840.0)
No Data: -9999.0
Data type: float32


In [14]:
#Función para obtner un dataframe que relacione la parcela con la elevación

def get_elevation(parcela, elev_path):
    with rasterio.open(elev_path) as src:
        
        parcela_raster = parcela.to_crs(src.crs) #Pasamos la parcela a mismo formato del raster
        
        #Obtenemos el poligono de la parcela
        geom_parcela = parcela_raster.geometry.iloc[0]
        geom = [mapping(geom_parcela)]
        
        #Recortamos la parcela alrededor del raster
        parcela_values, transform = mask(src, geom, crop=True, filled=False, all_touched=True)
        
        #Obtenemos el area de la parcela y obtenemos el area que tiene cada pixel dentro de la parcel
        area_total = geom_parcela.area
        area_valida = 0
        average_sum = 0
        
        for i in range(parcela_values.shape[1]):
            for j in range(parcela_values.shape[2]):
                
                if parcela_values.mask[0, i, j]:
                    continue #Ignoramos los No Data
                    
                #Obtemos el valor del raster
                elevacion = parcela_values[0, i, j]
                
                #Valores del pixel
                x_left, y_up = xy(transform, i, j, offset="ul")
                x_right, y_down = xy(transform, i, j, offset="lr")
                
                # Geometría del píxel
                pixel = box(x_left, y_down, x_right, y_up)
                
                #Obtenemos la parte del pixel que pertenece a la parcela
                intersection = pixel.intersection(geom_parcela)
                area_intersection = intersection.area
                area_valida += area_intersection
                
                average_sum += (float(elevacion) * area_intersection)
        
        # Media ponderada
        if area_valida > 0:
            elevacion_media = average_sum / area_valida
        else:
            elevacion_media = np.nan
        
        # Cobertura válida
        cobertura = area_valida / area_total
        
        return {
            "area_total_m2": area_total,
            "area_valida_m2": area_valida,
            "cobertura_elevacion": cobertura,
            "cobertura_elevacion_pct": cobertura * 100,
            "elevacion_media": elevacion_media
        } 

In [15]:
#Para las 197 parcelas

resultados_elevacion = []

for _, parcela in parcelas_gdf.iterrows():

    resultado = get_elevation(
        gpd.GeoDataFrame(
            [parcela],
            crs=parcelas_gdf.crs
        ),
        elev_path
    )
    
    resultado["ID_POLIGON"] = parcela["ID_POLIGON"]
    
    resultados_elevacion.append(resultado)

In [16]:
df_elevacion = pd.DataFrame(resultados_elevacion)

df_elevacion = df_elevacion[
    [
        "ID_POLIGON",
        "area_total_m2",
        "area_valida_m2",
        "cobertura_elevacion",
        "cobertura_elevacion_pct",
        "elevacion_media"
    ]
]

df_elevacion.head()

,ID_POLIGON,area_total_m2,area_valida_m2,cobertura_elevacion,cobertura_elevacion_pct,elevacion_media
0,AGC_196,85130.668788,85130.668788,1.000000,100.000000,2543.659752
1,AGC_044,6760.402971,6760.402971,1.000000,100.000000,2529.993625
2,AGC_020,82856.226598,81059.998963,0.978321,97.832115,2528.924828
3,AGC_174,68423.862739,68423.862739,1.000000,100.000000,2546.616547
4,AGC_083,28466.739798,28466.739798,1.000000,100.000000,2564.037732


In [17]:
df_elevacion["elevacion_media"].describe()

count     197.000000
mean     2667.102482
std        92.283855
min      2496.648468
25%      2611.351962
50%      2638.902931
75%      2726.683106
max      2906.039520
Name: elevacion_media, dtype: float64

In [18]:
print("Número de parcelas:", len(df_elevacion))
print("Elevaciones NaN:", df_elevacion["elevacion_media"].isna().sum())
print("Coberturas NaN:", df_elevacion["cobertura_elevacion_pct"].isna().sum())

Número de parcelas: 197
Elevaciones NaN: 0
Coberturas NaN: 0


In [19]:
print(
    df_elevacion[
        df_elevacion["cobertura_elevacion_pct"] < 99.999
    ][
        ["ID_POLIGON", "cobertura_elevacion_pct", "elevacion_media"]
    ]
)


    ID_POLIGON  cobertura_elevacion_pct  elevacion_media
2      AGC_020                97.832115      2528.924828
163    AGC_023                95.899497      2502.457298
175    AGC_170                75.938015      2501.227723
178    AGC_164                95.061206      2496.648468


### Raster Pendiente

In [4]:
pend_path = '../../DataSet/DATASET_RETO_AGRO_2026/Reto_AgroCebada_Topografia_INEGI_CEM4/Reto_AgroCebada_Topografia_INEGI_CEM4/Reto_AgroCebada_Topografia_INEGI_CEM4/Pendiente_INEGI_CEM4_120m_grados.tif'

In [5]:
def get_slope(parcela, raster_path):
    
    with rasterio.open(raster_path) as src:
        
        # Reproyectar parcela al CRS del raster
        parcela_raster = parcela.to_crs(src.crs)
        
        # Geometría de la parcela
        geom_parcela = parcela_raster.geometry.iloc[0]
        geom = [mapping(geom_parcela)]
        
        # Recortar raster alrededor de la parcela
        valores, transform = mask(
            src,
            geom,
            crop=True,
            filled=False,
            all_touched=True
        )
        
        # Área total
        area_total = geom_parcela.area
        
        area_valida = 0
        suma_ponderada = 0
        
        # Recorrer píxeles
        for fila in range(valores.shape[1]):
            for columna in range(valores.shape[2]):
                
                # Ignorar NoData
                if valores.mask[0, fila, columna]:
                    continue
                
                # Valor de pendiente
                pendiente = valores[0, fila, columna]
                
                # Coordenadas del píxel
                x_izq, y_sup = rasterio.transform.xy(
                    transform,
                    fila,
                    columna,
                    offset="ul"
                )
                
                x_der, y_inf = rasterio.transform.xy(
                    transform,
                    fila,
                    columna,
                    offset="lr"
                )
                
                # Geometría del píxel
                pixel = box(
                    x_izq,
                    y_inf,
                    x_der,
                    y_sup
                )
                
                # Intersección con la parcela
                interseccion = pixel.intersection(geom_parcela)
                
                area_interseccion = interseccion.area
                
                # Área válida
                area_valida += area_interseccion
                
                # Pendiente × área
                suma_ponderada += (
                    float(pendiente) *
                    area_interseccion
                )
        
        # Media ponderada
        if area_valida > 0:
            pendiente_media = suma_ponderada / area_valida
        else:
            pendiente_media = np.nan
        
        # Cobertura
        cobertura = area_valida / area_total
        
        return {
            "area_total_m2": area_total,
            "area_valida_m2": area_valida,
            "cobertura_pendiente": cobertura,
            "cobertura_pendiente_pct": cobertura * 100,
            "pendiente_media": pendiente_media
        }

In [6]:
parcela_001 = parcelas_gdf[
    parcelas_gdf["ID_POLIGON"] == "AGC_001"
].copy()

resultado_001_pendiente = get_slope(
    parcela_001,
    pend_path
)

print(resultado_001_pendiente)

{'area_total_m2': 75685.78283383907, 'area_valida_m2': 75685.78283386417, 'cobertura_pendiente': 1.0000000000003317, 'cobertura_pendiente_pct': 100.00000000003317, 'pendiente_media': 3.039547531881103}


In [7]:
resultados_pendiente = []

for _, parcela in parcelas_gdf.iterrows():

    resultado = get_slope(gpd.GeoDataFrame([parcela], crs=parcelas_gdf.crs),pend_path)
    
    resultado["ID_POLIGON"] = parcela["ID_POLIGON"]
    
    resultados_pendiente.append(resultado)

In [8]:
df_pendiente = pd.DataFrame(resultados_pendiente)

df_pendiente = df_pendiente[
    [
        "ID_POLIGON",
        "area_total_m2",
        "area_valida_m2",
        "cobertura_pendiente",
        "cobertura_pendiente_pct",
        "pendiente_media"
    ]
]

In [9]:
print(df_pendiente.head())

  ID_POLIGON  area_total_m2  area_valida_m2  cobertura_pendiente  \
0    AGC_196   85130.668788    85130.668788             1.000000   
1    AGC_044    6760.402971     6626.924578             0.980256   
2    AGC_020   82856.226598    54063.973641             0.652503   
3    AGC_174   68423.862739    68423.862739             1.000000   
4    AGC_083   28466.739798    28466.739798             1.000000   

   cobertura_pendiente_pct  pendiente_media  
0               100.000000         1.740829  
1                98.025585         0.348862  
2                65.250345         0.561396  
3               100.000000         2.284557  
4               100.000000         2.368227  


### Elevación + Pendiente

In [20]:
df_topografia = df_elevacion.merge(df_pendiente, on='ID_POLIGON', how='inner', suffixes=('_elevacion', '_pendiente'))

df_topografia = df_topografia[
    [
        "ID_POLIGON",
        "elevacion_media",
        "cobertura_elevacion_pct",
        "pendiente_media",
        "cobertura_pendiente_pct"
    ]
]

In [21]:
df_topografia.head()

,ID_POLIGON,elevacion_media,cobertura_elevacion_pct,pendiente_media,cobertura_pendiente_pct
0,AGC_196,2543.659752,100.000000,1.740829,100.000000
1,AGC_044,2529.993625,100.000000,0.348862,98.025585
2,AGC_020,2528.924828,97.832115,0.561396,65.250345
3,AGC_174,2546.616547,100.000000,2.284557,100.000000
4,AGC_083,2564.037732,100.000000,2.368227,100.000000


## Datos climatológicos